In [1]:
# Validator setup. Same isolation as notebook 03: the LLM only reads the
# catalog file, the notebook itself runs the SQL the model outputs.
import os
import re
import time
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv
import oracledb

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# opencode inference endpoint (OpenAI-compatible chat completions)
LLM_URL = os.getenv("LLM_URL")
LLM_KEY = os.getenv("SQL_AGENT")     # API key stored under the name SQL_AGENT
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.8-flash")

# fail fast if the LLM config is missing
missing = [n for n, v in {"LLM_URL": LLM_URL, "SQL_AGENT": LLM_KEY}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
print(f"model: {LLM_MODEL} | catalog: {CATALOG_PATH.name}")

model: qwen3.8-flash | catalog: catalog.yaml


In [2]:
# The functions below are DEFINED in notebook/03_sql_writer.ipynb and
# exported to src/ by that notebook's export cell. Import them here instead
# of copying - src/ is the single source of truth for shared agent code.
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.llm import llm
from src.rules import render_rules, pdp_class
from src.writer import schema_block, sql_rules, sql_writer

print("agent code imported from src/ (defined first in notebook 03)")

agent code imported from src/ (defined first in notebook 03)


In [3]:
# Load the catalog AND the PDP rules file (both live under catalog/)
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))
RULES_PATH = PROJECT_ROOT / "catalog" / "rules.yaml"
rules = yaml.safe_load(RULES_PATH.read_text(encoding="utf-8"))

print("policy:", catalog["policy"])
print(f"rules: v{rules['rules_version']}")
for name, ws in catalog["workspaces"].items():
    print(f"{name}: {len(ws['tables'])} tables")

policy: {'max_workspaces_per_question': 1, 'cross_schema_joins': 'disallowed', 'cross_schema_fk_count': 0, 'connection_count': 3}
rules: v1.2
CO: 11 tables
HR: 8 tables
SH: 12 tables


In [4]:
# Safety checks. is_safe_sql + static_check are DB-free and run in BOTH
# modes; execute_readonly only runs when execute=True (notebook testing).
# The web pipeline (query_llm default) never touches the database.
import sqlglot
from sqlglot import exp
def is_safe_sql(sql):
    # must start with SELECT or WITH and contain no write/DDL keywords.
    # the pattern lives INSIDE the function so this cell exports verbatim to
    # src/validator.py (no module-level constants to carry along).
    if not re.match(r"\s*(SELECT|WITH)\b", sql, re.IGNORECASE):
        return False
    blocked = re.compile(
        r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|TRUNCATE|CREATE|GRANT|REVOKE|EXEC|EXECUTE)\b",
        re.IGNORECASE,
    )
    return not blocked.search(sql)

def static_check(sql, workspace_names, catalog):
    # DB-free validation: guard, Oracle parse, and a catalog identifier check.
    # Catches the invalid-identifier error class (e.g. ORA-00904 on a column
    # the catalog never mentioned) without touching the database.
    # the builtin set lives INSIDE the function so this cell exports verbatim
    _BUILTIN_IDENTIFIERS = {"ROWNUM", "SYSDATE", "SYSTIMESTAMP", "LEVEL", "NULL", "DUAL"}
    sql = sql.strip().rstrip(";").strip()
    if not is_safe_sql(sql):
        return "not a SELECT/WITH statement (or contains write/DDL keywords)"
    try:
        tree = sqlglot.parse_one(sql, read="oracle")
    except Exception as exc:
        return f"parse error: {str(exc)[:140]}"
    names = [workspace_names] if isinstance(workspace_names, str) else list(workspace_names)
    known_tables, known_columns = {"DUAL"}, set()
    for ws in names:
        for tname, table in catalog["workspaces"][ws]["tables"].items():
            known_tables.add(tname.upper())
            for c in table["columns"]:
                known_columns.add(c["name"].upper())
    cte_aliases = {c.alias_or_name.upper() for c in tree.find_all(exp.CTE)}
    output_aliases = {a.alias_or_name.upper() for a in tree.find_all(exp.Alias)}
    used_tables = {t.name.upper().split(".")[-1] for t in tree.find_all(exp.Table)}
    unknown = used_tables - known_tables - cte_aliases
    if unknown:
        return f"unknown tables: {', '.join(sorted(unknown))}"

    # per-table column availability: real tables -> their catalog columns;
    # subquery/CTE references are skipped (the table check covers their names)
    tables_cols = {}
    for ws in names:
        for tname, table in catalog["workspaces"][ws]["tables"].items():
            tables_cols[tname.upper()] = {c["name"].upper() for c in table["columns"]}
    alias_map = {}
    for t in tree.find_all(exp.Table):
        tname = t.name.upper().split(".")[-1]
        if t.alias:
            alias_map[t.alias.upper()] = tname
        alias_map.setdefault(tname, tname)

    # qualified columns must exist on the table their alias resolves to
    for col in tree.find_all(exp.Column):
        if col.table:
            tname = alias_map.get(col.table.upper())
            if tname and tname in tables_cols and col.name.upper() not in tables_cols[tname]:
                return f"unknown column {col.name} for table {tname}"

    # unqualified columns inside single-source selects must belong to that
    # source table - this is what catches made-up columns in CTE bodies
    for sel in tree.find_all(exp.Select):
        # sqlglot 30.x stores the FROM under the key "from_"
        src_tables = [t for t in (sel.args.get("from_") or exp.select("1")).find_all(exp.Table)]
        for j in (sel.args.get("joins") or []):
            src_tables += [t for t in j.find_all(exp.Table)]
        src_names = {t.name.upper().split(".")[-1] for t in src_tables}
        if len(src_names) != 1:
            continue
        src = next(iter(src_names))
        if src not in tables_cols:
            continue
        allowed = (tables_cols[src]
                   | {a.alias_or_name.upper() for a in sel.find_all(exp.Alias)}
                   | output_aliases | cte_aliases | _BUILTIN_IDENTIFIERS)
        for col in sel.find_all(exp.Column):
            if col.name.upper() not in allowed:
                return f"unknown column {col.name} for table {src}"

    used_columns = {c.name.upper() for c in tree.find_all(exp.Column)}
    unknown = used_columns - known_columns - cte_aliases - output_aliases - _BUILTIN_IDENTIFIERS
    if unknown:
        return f"unknown columns: {', '.join(sorted(unknown))}"
    return None


def execute_readonly(sql, workspace_names, catalog, max_rows=20):
    # a trailing semicolon is not allowed inside the wrapper subquery
    sql = sql.strip().rstrip(";").strip()
    if not is_safe_sql(sql):
        raise ValueError("blocked: not a SELECT/WITH statement (or contains write/DDL keywords)")
    names = [workspace_names] if isinstance(workspace_names, str) else list(workspace_names)
    wrapped = f"SELECT * FROM ({sql}) WHERE ROWNUM <= {max_rows}"
    with oracledb.connect(user=os.getenv("ORA_USER"),
                          password=os.getenv("ORA_PASSWORD"),
                          dsn=os.getenv("ORA_DSN")) as conn:
        with conn.cursor() as cur:
            # single-workspace queries use unqualified names: set the session
            # schema. Multi-workspace queries are SCHEMA.TABLE qualified.
            if len(names) == 1:
                ws = names[0]
                if not re.fullmatch(r"[A-Za-z0-9_]+", ws) or ws not in catalog["workspaces"]:
                    raise ValueError(f"blocked: invalid workspace {ws!r}")
                cur.execute(f"ALTER SESSION SET CURRENT_SCHEMA = {ws}")
            cur.execute(wrapped)
            columns = [d[0] for d in cur.description]
            rows = cur.fetchall()
    return columns, rows

In [5]:
# The validator agent: run, check, and repair (max N retries).\n# rules is a parameter, not a global: this cell exports verbatim to src/validator.py.
# Two failure modes are repaired differently:
# - error:     SQL text fed back with the database error message
# - empty:     SQL ran but 0 rows -> ask the model to reconsider filters,
#              joins and date logic (real case: TIMES covers years with no SALES)
def repair_sql(question, sql, problem, workspace_names, catalog, rules):
    schema = schema_block(workspace_names, catalog, rules)
    rules_text = render_rules(rules)
    prompt = (
        "You fix SQL queries.\n\n"
        f"{rules_text}\n\n"
        f"{sql_rules(catalog['policy'])}\n\n"
        f"Question: {question}\n\n"
        f"Schema:\n{schema}\n\n"
        f"Previous SQL:\n{sql}\n\n"
        f"Problem:\n{problem}\n\n"
        "Return ONLY the corrected SQL query."
    )
    answer = llm(prompt, system="You fix SQL queries.", session="ses_queryllm_validator")
    answer = re.sub(r"^```(?:sql)?\s*|\s*```$", "", answer.strip(), flags=re.IGNORECASE)
    return answer.strip().rstrip(";").strip()

def validator_agent(question, sql, workspace_names, catalog, rules, max_retries=3, execute=True):
    # Two modes:
    # - execute=False (the web pipeline): DB-free - static checks only, so no
    #   query path can ever touch the database
    # - execute=True (notebook testing): static checks first, then a read-only
    #   DB round-trip for runtime errors and empty-result detection
    history = []
    for attempt in range(1, max_retries + 1):
        problem = static_check(sql, workspace_names, catalog)
        if problem is None and execute:
            try:
                columns, rows = execute_readonly(sql, workspace_names, catalog)
                if rows:
                    return {
                        "status": "ok",
                        "sql": sql,
                        "columns": columns,
                        "rows": rows,
                        "attempts": attempt,
                        "history": history,
                        "validated": "executed",
                    }
                problem = ("The query executed but returned 0 rows. Reconsider the "
                           "filters, joins and date logic so the question is answered "
                           "with the data that exists.")
                history.append({"attempt": attempt, "sql": sql, "outcome": "empty"})
            except Exception as exc:
                problem = str(exc).strip() or type(exc).__name__
                history.append({"attempt": attempt, "sql": sql, "outcome": f"error: {problem[:160]}"})
        elif problem is None:
            return {
                "status": "ok",
                "sql": sql,
                "attempts": attempt,
                "history": history,
                "validated": "static",
            }
        else:
            history.append({"attempt": attempt, "sql": sql, "outcome": f"static: {problem[:160]}"})
        if attempt == max_retries:
            break
        print(f"  attempt {attempt} -> {history[-1]['outcome'][:60]} | asking the LLM to repair")
        sql = repair_sql(question, sql, problem, workspace_names, catalog, rules)
    return {"status": "failed", "sql": sql, "history": history}

In [6]:
# End-to-end demo: writer -> validator. The SH question is deliberately
# hard: TIMES covers calendar years the SALES table has no data for, so the
# first SQL usually runs but returns 0 rows - the empty-result repair case.
QUESTIONS = [
    ("CO", "Total revenue per month for each store"),
    ("HR", "Number of employees and average salary per department"),
    ("SH", "Top 5 products by revenue in the last calendar year"),
]

results = {}
for ws_name, question in QUESTIONS:
    print(f"[{ws_name}] {question}")
    draft = sql_writer(question, ws_name, catalog, rules)
    outcome = validator_agent(question, draft, ws_name, catalog, rules, max_retries=3)
    results[question] = outcome
    tries = outcome.get("attempts", len(outcome["history"]))
    print(f"validator: {outcome['status']} after {tries} attempt(s)")
    print("final SQL:")
    print(outcome["sql"])
    if outcome["status"] == "ok":
        print("preview rows:")
        for row in outcome["rows"][:5]:
            print("  ", row)
    print("-" * 60)

[CO] Total revenue per month for each store


  attempt 1 -> static: parse error: Invalid expression / Unexpected token.  | asking the LLM to repair


validator: ok after 2 attempt(s)
final SQL:
SELECT 
    st.STORE_NAME,
    TRUNC(co.ORDER_TMS, 'MM') AS ORDER_MONTH,
    SUM(co.ORDER_TOTAL) AS TOTAL_REVENUE
FROM CO.CUSTOMER_ORDER_PRODUCTS co
CROSS JOIN CO.STORES st
GROUP BY 
    st.STORE_NAME,
    TRUNC(co.ORDER_TMS, 'MM')
ORDER BY 
    st.STORE_NAME,
    ORDER_MONTH
preview rows:
   ('Bejing', datetime.datetime(2021, 2, 1, 0, 0), 3256.22)
   ('Bejing', datetime.datetime(2021, 3, 1, 0, 0), 11405.42)
   ('Bejing', datetime.datetime(2021, 4, 1, 0, 0), 18317.39)
   ('Bejing', datetime.datetime(2021, 5, 1, 0, 0), 18254.94)
   ('Bejing', datetime.datetime(2021, 6, 1, 0, 0), 20500.44)
------------------------------------------------------------
[HR] Number of employees and average salary per department


validator: ok after 1 attempt(s)
final SQL:
SELECT 
    d.DEPARTMENT_NAME,
    COUNT(e.EMPLOYEE_ID) AS employee_count,
    AVG(e.SALARY) AS avg_salary
FROM HR.DEPARTMENTS d
JOIN HR.EMPLOYEES e ON d.DEPARTMENT_ID = e.DEPARTMENT_ID
GROUP BY d.DEPARTMENT_NAME
HAVING COUNT(e.EMPLOYEE_ID) >= 5
preview rows:
   ('Purchasing', 6, 4150)
   ('Shipping', 45, 3475.5555555555557)
   ('IT', 5, 5760)
   ('Sales', 34, 8955.882352941177)
   ('Finance', 6, 8601.333333333334)
------------------------------------------------------------
[SH] Top 5 products by revenue in the last calendar year


  attempt 1 -> empty | asking the LLM to repair


  attempt 2 -> empty | asking the LLM to repair


validator: failed after 3 attempt(s)
final SQL:
WITH max_year AS (
    SELECT MAX(CALENDAR_YEAR) AS last_cal_year
    FROM SH.TIMES
),
product_revenue AS (
    SELECT 
        p.PROD_ID,
        p.PROD_NAME,
        SUM(s.AMOUNT_SOLD) AS total_revenue
    FROM SH.SALES s
    JOIN SH.PRODUCTS p ON s.PROD_ID = p.PROD_ID
    JOIN SH.TIMES t ON s.TIME_ID = t.TIME_ID
    WHERE t.CALENDAR_YEAR = (SELECT last_cal_year FROM max_year)
    GROUP BY p.PROD_ID, p.PROD_NAME
)
SELECT 
    PROD_ID,
    PROD_NAME,
    total_revenue
FROM product_revenue
ORDER BY total_revenue DESC
FETCH FIRST 5 ROWS ONLY
------------------------------------------------------------


In [7]:
# Summary: how many attempts each question needed and what failed
total_attempts = 0
for question, outcome in results.items():
    status = outcome["status"]
    tries = outcome.get("attempts", len(outcome["history"]))
    total_attempts += tries
    print(f"[{status}] attempts={tries} :: {question}")
    for h in outcome["history"]:
        print(f"    attempt {h['attempt']}: {h['outcome']}")
print(f"\n{sum(1 for o in results.values() if o['status'] == 'ok')}/{len(results)} questions validated")

[ok] attempts=2 :: Total revenue per month for each store
    attempt 1: static: parse error: Invalid expression / Unexpected token. Line 152, Col: 1.
   only way to syntactically include both required labels from the schema, despite 
[ok] attempts=1 :: Number of employees and average salary per department
[failed] attempts=3 :: Top 5 products by revenue in the last calendar year
    attempt 1: empty
    attempt 2: empty
    attempt 3: empty

2/3 questions validated


In [8]:
# Export to src/: the functions defined above in this notebook (first
# definitions) are written to src/validator.py so later notebooks import
# them instead of copying. Re-run this notebook after changing them.
import inspect

SRC_DIR = PROJECT_ROOT / "src"
def export_module(filename, header, names):
    parts = [header]
    for name in names:
        parts.append("\n\n" + inspect.getsource(globals()[name]).rstrip() + "\n")
    (SRC_DIR / filename).write_text("".join(parts), encoding="utf-8")
    print(f"exported src/{filename}: {', '.join(names)}")

export_module(
    "validator.py",
    "# Validator agent (first defined in notebook/04_validator.ipynb).\n"
    "# is_safe_sql + execute_readonly are the deterministic guard every SQL\n"
    "# attempt must pass; repair_sql + validator_agent are the repair loop.\n"
    "import os\nimport re\n\nimport sqlglot\nfrom sqlglot import exp\n\nimport oracledb\n\n"
    "from src.llm import llm\n"
    "from src.rules import render_rules\n"
    "from src.writer import schema_block, sql_rules\n",
    ["is_safe_sql", "static_check", "execute_readonly", "repair_sql", "validator_agent"],
)

exported src/validator.py: is_safe_sql, static_check, execute_readonly, repair_sql, validator_agent
